# Is this text usual, or unusual?

Take any piece of ordinary English writing — a novel, a newspaper
article, an email — and count how often each word appears. Long before
you look at what it's actually *about*, the shape of that count already
tells you something: a handful of tiny words (*the*, *to*, *and*) show
up constantly, and the vast majority of words show up once or twice and
never again.

That shape has a name — **Zipf's law** — and it's remarkably
consistent across ordinary English text. This notebook checks whether
one real book actually follows it, using nothing but counting and a
chart. The book is *Pride and Prejudice* (Jane Austen, 1813) — long out
of copyright, and shipped with dewlab as `data/pride-and-prejudice.txt`
via [Project Gutenberg](https://www.gutenberg.org/ebooks/1342).

## Getting the text in

`load_csv()` only handles CSV files — this is a plain text file, so it
needs `pyfetch` directly instead, the same tool `load_csv()` uses
internally.

In [ ]:
from pyodide.http import pyfetch

response = await pyfetch("../data/pride-and-prejudice.txt")
raw_text = await response.string()
print(f"{len(raw_text):,} characters")

## Cleaning it up first

Real text data is never quite ready to use. Project Gutenberg wraps
every book it publishes in a standard header and footer — licence
information, not part of the novel — and this file is no exception.
Skip this step and your word counts would include phrases like "Project
Gutenberg" and "United States" dozens of times, which have nothing to
do with Jane Austen.

In [ ]:
start_marker = "*** START OF THIS PROJECT GUTENBERG EBOOK"
end_marker = "*** END OF THIS PROJECT GUTENBERG EBOOK"

start = raw_text.find(start_marker)
start = raw_text.find("\n", start) + 1   # skip past the marker line itself
end = raw_text.find(end_marker)

novel_text = raw_text[start:end]
print(f"{len(novel_text):,} characters of actual novel")

## Counting words

A "word" here means a run of letters — `re.findall(r"[a-zA-Z']+", ...)`
pulls out every such run, lower-cased so `"The"` and `"the"` count as
the same word. Punctuation, chapter numbers, and blank lines all fall
away on their own; none of them match that pattern.

In [ ]:
import re
from collections import Counter

words = re.findall(r"[a-zA-Z']+", novel_text.lower())
word_counts = Counter(words)

print(f"{len(words):,} words in total")
print(f"{len(word_counts):,} of them are distinct words")

Already a striking gap — tens of thousands of words on the page, only a
few thousand distinct ones. Most of that difference is repetition: the
same small set of words, over and over. `Counter.most_common()` shows
exactly which ones.

In [ ]:
import pandas as pd

show_table(pd.DataFrame(word_counts.most_common(15), columns=["word", "count"]))

None of those fifteen words are nouns, and none of them are specific to
Austen, or to this book, or arguably to any particular story at all —
*the*, *to*, *of*, *and*, *her*. They're the connective tissue of
English sentences, and they dominate the count precisely because every
sentence needs them, regardless of what it's actually saying.

## The pattern: rank against frequency

Number every word by rank — 1st most common, 2nd most common, and so
on — and plot rank against how often that word appeared.

In [ ]:
import matplotlib.pyplot as plt

ranked = word_counts.most_common()          # already sorted, most common first
ranks = range(1, len(ranked) + 1)
frequencies = [count for word, count in ranked]

plt.plot(ranks, frequencies)
plt.xlabel("Rank")
plt.ylabel("Number of times the word appears")
plt.title("Word frequency in Pride and Prejudice")
plt.show()

A steep drop, then a long, flat-looking tail — the first few words
towering over almost everything else. That shape alone is a clue, but
the real test needs a different kind of chart.

In [ ]:
plt.plot(ranks, frequencies)
plt.xscale("log")
plt.yscale("log")
plt.xlabel("Rank (log scale)")
plt.ylabel("Frequency (log scale)")
plt.title("Same data, log-log axes")
plt.show()

**This is the test.** Zipf's law predicts that a word's frequency is
roughly proportional to $\frac{1}{\text{rank}}$ — the 2nd most common
word appears about half as often as the 1st, the 3rd about a third as
often, and so on. On a log-log chart, that specific relationship draws
a straight line. If the line above looks close to straight — especially
through the first few hundred words — this real, ordinary novel is
behaving exactly the way Zipf's law says ordinary English should.

You can check the same claim numerically, not just by eye: multiply
each rank by its frequency, and Zipf's law predicts something close to
a constant.

In [ ]:
for rank in [1, 2, 3, 5, 10, 20, 50, 100]:
    word, count = ranked[rank - 1]
    print(f"rank {rank:>4}  word {word!r:>8}  count {count:>5}  rank x count = {rank * count}")

Not perfectly constant — real text never fits a mathematical law
exactly — but in the same rough range for a wide stretch of ranks,
rather than climbing or falling steadily. That's what "roughly Zipfian"
looks like in practice.

## Your turn: is it always true?

Zipf's law is a statement about *ordinary* text. The interesting
question is what happens to text that *isn't* ordinary in some way.
Pick one and try it:

- **A different language.** If dewlab has another text file available,
  or you paste in a paragraph of your own in a language other than
  English, does the same log-log plot still look roughly straight?
- **A word list, not prose.** Build a "text" that's just the same
  handful of words repeated in a pattern rather than a real sentence —
  does *that* still look Zipfian, or does the shape break?
- **Your own writing.** Paste a paragraph or two of something you wrote
  — an essay, a text conversation, anything — into a Python string and
  run it through the same pipeline: `re.findall`, `Counter`,
  log-log plot. Real, unscripted writing almost always turns out
  Zipfian too, even though nobody sits down intending to write that
  way.

(`hint:` everything after `words = re.findall(...)` above works on any
string called `novel_text` — swap in a shorter one of your own and
re-run from there.)

In [ ]:
# Your turn — a different text, through the same pipeline.


## Looking back

Nobody designs a novel to obey Zipf's law. Nobody designs *any* text
that way — it falls out on its own, from nothing more than "a few words
are useful in almost every sentence, and most words are only useful
sometimes." That a strict mathematical pattern emerges from something
as unplanned as ordinary writing is what makes it worth noticing: a
regularity hiding in something that feels like it should have none.

**Source**: Jane Austen, *Pride and Prejudice* (1813), via
[Project Gutenberg](https://www.gutenberg.org/ebooks/1342) — public
domain in the United States. If you redistribute this text, Project
Gutenberg's licence asks that its standard header stay attached to it;
`data/pride-and-prejudice.txt` keeps it for exactly that reason, which
is also why this notebook strips it out again before counting.